## Simulations 3a

In [ ]:
import sys
sys.path.insert(0, '..')

In [ ]:
from scm import RSCGenerator, SCMSimulator, RandomSeeding
from scm.analysis import mf_rho, steady_state_rho
import numpy as np

def sim_rho(lam_d,
            N = 2000,
            k_avg = 20,
            k_d_avg = 6,
            mu = 0.1,
            rho_0 = 0.4,
            t_max = 2000,
            t_avg = 100,
            lambda_vals = np.linspace(0.0, 2.5, 30)
            ):

    generator = RSCGenerator(k_avg=k_avg, k_delta_avg=k_d_avg, N=N)
    links, triangles = generator.generate(seed=2025)
    rho_stars = []
    beta_d = (lam_d * mu) / k_d_avg

    for lam in lambda_vals:
        beta = (lam * mu) / k_avg
        
        if beta > 1.0 or beta_d > 1.0:
            raise ValueError("Calculated beta exceeds 1.0. Decrease mu.")
            
        seeder = RandomSeeding(N, rho_0)
        initial_infected = seeder.seed()
        
        sim = SCMSimulator(
            links=links, 
            triangles=triangles, 
            initial_infected=initial_infected, 
            beta=beta, 
            beta_delta=beta_d, 
            mu=mu
        )
        
        rho_history = sim.run(t_max)
        rho_star = steady_state_rho(rho_history, t_avg)
        rho_stars.append(rho_star)
    return np.array(rho_stars)

## Network Statistics

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

N = 2000
k_avg = 20
k_d_avg = 6

generator = RSCGenerator(k_avg=k_avg, k_delta_avg=k_d_avg, N=N)
links, triangles = generator.generate(seed=2025)

# Build networkx graph from the full edge set
G = nx.Graph()
G.add_nodes_from(range(N))
for i in range(N):
    for j in links[i]:
        if i < j:
            G.add_edge(i, j)

# --- Degree stats ---
degrees = np.array([G.degree(i) for i in range(N)])
triangle_counts = np.array([len(triangles[i]) for i in range(N)])

num_edges = G.number_of_edges()
num_triangles_explicit = sum(triangle_counts) // 3
num_triangles_graph = sum(nx.triangles(G, i) for i in range(N)) // 3

print("=== Basic Counts ===")
print(f"Nodes: {N}")
print(f"Edges: {num_edges}")
print(f"Explicit 2-simplices: {num_triangles_explicit}")
print(f"Total graph triangles: {num_triangles_graph}")
print(f"Incidental triangles (from random edges): {num_triangles_graph - num_triangles_explicit}")
print(f"Edge density: {nx.density(G):.6f}")
print()

print("=== Degree (edges) ===")
print(f"Mean: {degrees.mean():.2f}  (target k_avg={k_avg})")
print(f"Std:  {degrees.std():.2f}")
print(f"Min:  {degrees.min()}  Max: {degrees.max()}")
print()

print("=== 2-Simplex Participation (per node) ===")
print(f"Mean: {triangle_counts.mean():.2f}  (target k_delta_avg={k_d_avg})")
print(f"Std:  {triangle_counts.std():.2f}")
print(f"Min:  {triangle_counts.min()}  Max: {triangle_counts.max()}")
print()

# --- Clustering coefficient (from networkx, counts ALL triangles in the edge graph) ---
clustering = np.array([nx.clustering(G, i) for i in range(N)])

print("=== Clustering Coefficient ===")
print(f"Mean: {clustering.mean():.4f}")
print(f"Std:  {clustering.std():.4f}")
print(f"Expected (ER): {k_avg / (N - 1):.4f}")
print()

# --- Edge-2-simplex overlap ---
triangle_edge_set = set()
for i in range(N):
    for j, k in triangles[i]:
        for edge in [(min(i,j), max(i,j)), (min(j,k), max(j,k)), (min(i,k), max(i,k))]:
            triangle_edge_set.add(edge)

print("=== Edge-2-Simplex Overlap ===")
print(f"Edges in at least one 2-simplex: {len(triangle_edge_set)} ({100*len(triangle_edge_set)/num_edges:.1f}%)")
print(f"Edges not in any 2-simplex: {num_edges - len(triangle_edge_set)} ({100*(num_edges - len(triangle_edge_set))/num_edges:.1f}%)")
print()

# --- Plots ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Degree distribution
axes[0].hist(degrees, bins=range(degrees.min(), degrees.max() + 2), 
             density=True, alpha=0.7, edgecolor='black', linewidth=0.5)
axes[0].axvline(k_avg, color='red', ls='--', label=f'target $\\langle k \\rangle$={k_avg}')
axes[0].set_xlabel('Degree $k$')
axes[0].set_ylabel('Density')
axes[0].set_title('Degree Distribution')
axes[0].legend()

# Triangle participation distribution
axes[1].hist(triangle_counts, bins=range(triangle_counts.min(), triangle_counts.max() + 2),
             density=True, alpha=0.7, edgecolor='black', linewidth=0.5, color='tab:orange')
axes[1].axvline(k_d_avg, color='red', ls='--', label=f'target $\\langle k_\\Delta \\rangle$={k_d_avg}')
axes[1].set_xlabel('Triangle count $k_\\Delta$')
axes[1].set_ylabel('Density')
axes[1].set_title('2-Simplex Participation Distribution')
axes[1].legend()

# Clustering coefficient distribution
axes[2].hist(clustering[degrees >= 2], bins=50, density=True, alpha=0.7, 
             edgecolor='black', linewidth=0.5, color='tab:green')
axes[2].axvline(clustering.mean(), color='red', ls='--', label=f'mean={clustering.mean():.4f}')
axes[2].set_xlabel('Local clustering $C_i$')
axes[2].set_ylabel('Density')
axes[2].set_title('Clustering Coefficient Distribution')
axes[2].legend()

plt.tight_layout()
plt.savefig('../figures/network_stats.png', dpi=300)
plt.show()

# --- Degree vs triangle participation scatter ---
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(degrees, triangle_counts, alpha=0.15, s=8)
ax.set_xlabel('Degree $k$')
ax.set_ylabel('Triangle participation $k_\\Delta$')
ax.set_title('Degree vs 2-Simplex Participation')
plt.tight_layout()
plt.savefig('../figures/degree_vs_triangles.png', dpi=300)
plt.show()

## Plotting

In [ ]:
import matplotlib.pyplot as plt

lambdas_mf = np.linspace(0.01, 2.5, 1000)
mf_rho_0 = mf_rho(lambdas_mf, 0.0)
mf_rho_08 = mf_rho(lambdas_mf, 0.8)
mf_rho_25 = mf_rho(lambdas_mf, 2.5)

# Simulation Curves
lambdas_sim = np.linspace(0.0, 2.5, 30)
sim_rho_0 = sim_rho(lam_d=0.0, lambda_vals=lambdas_sim)
sim_rho_08 = sim_rho(lam_d=0.8, lambda_vals=lambdas_sim)
sim_rho_25 = sim_rho(lam_d=2.5, lambda_vals=lambdas_sim)

# Plot
plt.figure(figsize=(9, 6))
plt.plot(lambdas_mf, mf_rho_0, color='orange', lw=2.5)
plt.plot(lambdas_mf, mf_rho_08, color='orange', lw=2.5)
plt.plot(lambdas_mf, mf_rho_25, color='orange', lw=2.5)

plt.plot(lambdas_sim, sim_rho_0, color='blue', lw=2.5, label=r'$\lambda_\Delta = 0.0$')
plt.plot(lambdas_sim, sim_rho_08, color='green', lw=2.5, label=r'$\lambda_\Delta = 0.8$')
plt.plot(lambdas_sim, sim_rho_25, color='black', lw=2.5, label=r'$\lambda_\Delta = 2.5$')

plt.xlim(0, 2.2)
plt.ylim(0, 0.8)
plt.xlabel(r'Rescaled link infectivity, $\lambda$', fontsize=12)
plt.ylabel(r'Stationary density of infected nodes, $\rho^*$', fontsize=12)
plt.legend(loc='lower right', framealpha=1, edgecolor='black')
plt.grid(alpha=0.3)
plt.tight_layout()

# Save the plot
plt.savefig('../figures/analytical_curves.png', dpi=300)
plt.show()

## Simulations 3b

In [8]:
def sim_rho_history(lam=0.75, 
                    lam_d=2.5,
                    N=4000,
                    k_avg=20,
                    k_d_avg=6,
                    mu=0.1,
                    t_max=500,
                    rho_0_vals=None):
    
    if rho_0_vals is None:
        rho_0_vals = np.concatenate((
            np.linspace(0.02, 0.18, 8),
            np.linspace(0.22, 0.95, 15)
        ))

    generator = RSCGenerator(k_avg=k_avg, k_delta_avg=k_d_avg, N=N)
    links, triangles = generator.generate(seed=2026)
    
    histories = []
    beta = (lam * mu) / k_avg
    beta_d = (lam_d * mu) / k_d_avg

    if beta > 1.0 or beta_d > 1.0:
        raise ValueError("Calculated beta exceeds 1.0. Decrease mu.")

    for rho_0 in rho_0_vals:
        seeder = RandomSeeding(N, rho_0)
        initial_infected = seeder.seed()
        
        sim = SCMSimulator(
            links=links, 
            triangles=triangles, 
            initial_infected=initial_infected, 
            beta=beta, 
            beta_delta=beta_d, 
            mu=mu
        )
        
        rho_history = sim.run(t_max)
        histories.append((rho_0, rho_history))
        
    return histories

## Plotting

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors

# Simulation
lam_fixed = 0.75
lam_d_fixed = 2.5
t_max = 500
histories = sim_rho_history(lam=0.75, lam_d=2.5, t_max=500)

# Analytical unstable branch
discriminant = (lam_fixed - lam_d_fixed)**2 - 4 * lam_d_fixed * (1 - lam_fixed)
rho_2_minus = (lam_d_fixed - lam_fixed - np.sqrt(discriminant)) / (2 * lam_d_fixed)

# Setup plotting
fig, ax = plt.subplots(figsize=(9, 6))
norm = colors.Normalize(vmin=0, vmax=1)
cmap = cm.viridis

# Plot each history
for rho_0, rho_history in histories:
    color = cmap(norm(rho_0))
    ax.plot(range(len(rho_history)), rho_history, color=color, alpha=0.8, lw=1.5)

# Plot the unstable branch
ax.axhline(y=rho_2_minus, color='grey', linestyle='-.', alpha=0.5, label=r'$\rho_{2-}^*(\lambda, \lambda_\Delta)$')

# Formatting
ax.set_xlim(0, t_max)
ax.set_ylim(-0.02, 1.0)
ax.set_xlabel('Time, t', fontsize=12)
ax.set_ylabel(r'Density of infected nodes, $\rho(t)$', fontsize=12)

# Colorbar mapping to rho_0
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='horizontal', fraction=0.046, pad=0.1)
cbar.set_label(r'Initial density of infected nodes, $\rho_0$', fontsize=10)

plt.legend(loc='center right')
plt.tight_layout()
plt.savefig('../figures/bistability_trajectories.png', dpi=300)
plt.show()